# 1. Các phương pháp chuẩn hóa

Chuẩn hóa (normalization/scaling) là bước biến đổi dữ liệu số về cùng thang đo để:
- Giảm hiện tượng đặc trưng có độ lớn quá chênh lệch nhau.
- Giúp mô hình học máy hội tụ nhanh hơn.
- Làm cho khoảng cách/tối ưu hóa ổn định hơn (đặc biệt với các mô hình dựa trên khoảng cách và gradient).

## 1.1 Min-Max Scaling

### Nguyên lí
Biến đổi dữ liệu tuyến tính từ miền giá trị gốc về một khoảng cố định, phổ biến là $[0,1]$.

### Công thức
Với khoảng đích $[0,1]$:
$$
x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

Tổng quát với khoảng đích $[a,b]$:
$$
x' = a + \frac{(x - x_{\min})(b-a)}{x_{\max} - x_{\min}}
$$

### Ưu điểm
- Dễ hiểu, dễ triển khai.
- Giữ được thứ tự tương đối giữa các điểm dữ liệu.
- Hữu ích khi thuật toán cần dữ liệu trong khoảng hữu hạn (ví dụ mạng nơ-ron với một số hàm kích hoạt).

### Hạn chế
- Rất nhạy với outlier: chỉ một giá trị cực trị có thể làm phần lớn dữ liệu bị nén.
- Khi dữ liệu mới vượt ngoài $[x_{\min}, x_{\max}]$ của tập huấn luyện, giá trị biến đổi có thể ra ngoài khoảng mong muốn.

## 1.2 Z-score Standardization

### Nguyên lí
Đưa dữ liệu về dạng có trung bình gần 0 và độ lệch chuẩn gần 1.

### Công thức
$$
z = \frac{x - \mu}{\sigma}
$$
Trong đó:
- $\mu$: trung bình của đặc trưng trên tập huấn luyện.
- $\sigma$: độ lệch chuẩn của đặc trưng trên tập huấn luyện.

### Ưu điểm
- Phù hợp với nhiều thuật toán tuyến tính, SVM, Logistic Regression, PCA.
- Không ép dữ liệu vào khoảng cố định, nên vẫn phản ánh mức độ "xa trung bình".
- Thường hoạt động tốt khi phân phối dữ liệu gần chuẩn.

### Hạn chế
- Nhạy với outlier do $\mu$ và $\sigma$ bị ảnh hưởng mạnh bởi giá trị cực trị.
- Nếu dữ liệu lệch mạnh (skewed), hiệu quả chuẩn hóa có thể không tối ưu.

## 1.3 Robust Scaling

### Nguyên lí
Chuẩn hóa bằng trung vị và khoảng tứ phân vị để giảm ảnh hưởng của outlier.

### Công thức
$$
x' = \frac{x - Q_2}{IQR}, \quad IQR = Q_3 - Q_1
$$
Trong đó:
- $Q_2$: trung vị (median).
- $Q_1, Q_3$: tứ phân vị thứ nhất và thứ ba.

### Ưu điểm
- Bền vững trước outlier hơn Min-Max và Z-score.
- Thích hợp cho dữ liệu có phân phối lệch hoặc chứa nhiều nhiễu.

### Hạn chế
- Không đưa dữ liệu về khoảng cố định.
- Nếu dữ liệu có rất nhiều giá trị trùng nhau quanh median, việc scale có thể kém phân tách.

## 1.4 Quantile Transform

### Nguyên lí
Dựa trên phân phối tích lũy thực nghiệm (ECDF), ánh xạ dữ liệu gốc sang một phân phối mục tiêu.

Bước chung:
1. Tính bách phân vị của mỗi giá trị: $u = \hat{F}(x)$ với $u \in (0,1)$.
2. Ánh xạ $u$ sang phân phối đầu ra mong muốn.

### (a) Output phân phối Uniform
#### Công thức
$$
x' = u = \hat{F}(x)
$$
Kết quả nằm gần trong khoảng $[0,1]$ và có dạng gần đều.

#### Ưu điểm
- Giảm ảnh hưởng của outlier rất mạnh.
- Đưa dữ liệu về cùng một phân phối rõ ràng, thuận lợi cho một số mô hình.

#### Hạn chế
- Biến đổi phi tuyến nên có thể làm thay đổi khoảng cách gốc giữa các điểm dữ liệu.
- Có thể làm mất ý nghĩa tỉ lệ ban đầu của đặc trưng.

### (b) Output phân phối Normal
#### Công thức
$$
x' = \Phi^{-1}(u) = \Phi^{-1}(\hat{F}(x))
$$
Trong đó $\Phi^{-1}$ là hàm nghịch đảo CDF của phân phối chuẩn tắc.

#### Ưu điểm
- Tạo đặc trưng gần phân phối chuẩn, hữu ích cho các phương pháp giả định Gaussian.
- Giảm ảnh hưởng outlier tốt hơn Z-score trong nhiều trường hợp lệch mạnh.

#### Hạn chế
- Cũng là biến đổi phi tuyến, có thể làm méo quan hệ tuyến tính ban đầu.
- Với dữ liệu mới ngoài miền quan sát, phép ánh xạ có thể bão hòa ở hai đầu phân phối.


## 2. Bảng so sánh các phương pháp chuẩn hóa

| Phương pháp | Nguyên lí | Độ phức tạp (fit/transform) | Giả định chính | Phù hợp dữ liệu | Công thức tính |
|---|---|---|---|---|---|
| **Min-Max Scaling** | Co giãn tuyến tính dữ liệu về khoảng cố định (thường $[0,1]$). | Thấp: fit $O(n)$, transform $O(n)$ cho mỗi đặc trưng. | Giá trị min/max trong train đại diện tốt cho dữ liệu thực tế; ít outlier mạnh. | Dữ liệu sạch, ít ngoại lệ, cần giá trị trong khoảng hữu hạn. | $x' = \frac{x-x_{\min}}{x_{\max}-x_{\min}}$ (hoặc tổng quát về $[a,b]$). |
| **Z-score Standardization** | Tịnh tiến theo trung bình và co giãn theo độ lệch chuẩn để dữ liệu có mean 0, std 1. | Thấp: fit $O(n)$, transform $O(n)$. | Phân phối không quá lệch; outlier không quá cực đoan. | Mô hình tuyến tính, SVM, PCA, Logistic Regression. | $z = \frac{x-\mu}{\sigma}$ |
| **Robust Scaling** | Chuẩn hóa bằng median và IQR để giảm tác động outlier. | Thấp-trung bình: fit $O(n\log n)$ (cần quantile), transform $O(n)$. | Median và IQR mô tả tốt trung tâm và độ phân tán của dữ liệu. | Dữ liệu có nhiều outlier hoặc lệch phân phối. | $x' = \frac{x-Q_2}{Q_3-Q_1}$ |
| **Quantile Transform (Uniform output)** | Ánh xạ theo ECDF về phân phối đều trong $[0,1]$. | Trung bình-cao: fit cần sắp xếp/ước lượng quantile ($O(n\log n)$), transform nội suy theo quantile. | Thứ hạng (rank) quan trọng hơn khoảng cách tuyệt đối; chấp nhận biến đổi phi tuyến. | Dữ liệu lệch mạnh, nhiều outlier; cần phân phối đầu ra đồng đều. | $u=\hat{F}(x),\; x'=u$ |
| **Quantile Transform (Normal output)** | Dùng ECDF lấy $u$, sau đó chiếu qua nghịch đảo CDF chuẩn để về gần Gaussian. | Trung bình-cao: tương tự quantile uniform + phép biến đổi $\Phi^{-1}$. | Muốn dữ liệu gần chuẩn; chấp nhận mất quan hệ tuyến tính gốc. | Dữ liệu lệch mạnh, mô hình hưởng lợi khi đầu vào gần chuẩn. | $u=\hat{F}(x),\; x' = \Phi^{-1}(u)$ |

**Ghi chú:**
- $n$ là số mẫu trên một đặc trưng.
- Với dữ liệu mới (test/inference), luôn dùng tham số đã fit từ tập train để tránh rò rỉ dữ liệu.